In [ ]:
import sys
# here, either reference to satclip repo goes, or can try and create git submodule 🤷‍♀️ 
# https://github.com/microsoft/satclip/tree/main
sys.path.append('/home/cbutsko/Desktop/cbutsko_experiments/satclip/satclip')

# general
import numpy as np
import pandas as pd
import itertools
import gc
import re
import torch
import glob
import rioxarray
import xarray as xr
import rasterio as rio
import os
import swifter

# modeling
from catboost import CatBoostClassifier, Pool
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report, precision_score, recall_score
from sklearn.model_selection import train_test_split
# from sklearn.utils.validation import check_is_fitted, check_array
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from scipy.signal import find_peaks

# plotting and output
from tqdm.auto import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from cbutsko_utils import plot_confusion_matrix
import warnings
warnings.filterwarnings(action='ignore')
tqdm.pandas()

# SatClip libs
from load import get_satclip
from cbutsko_utils import prepare_satclip_embeddings

# processing utils
from cbutsko_utils import patch2feats, process_raw_features_input_df

# hierarchical clsfc libs
from hiclass import LocalClassifierPerParentNode, LocalClassifierPerNode
from cbutsko_utils import CatBoostClassifierWrapper, LocalClassifierPerNodeWrapper, LocalClassifierPerParentNodeWrapper

In [ ]:
# read files with mappings to crop names and hierarchy

label_columns = ['cropland', 'landcover', 'cropgroup', 'croptype']

wc2ec_map = pd.read_csv('resources/wc2eurocrops_map.csv')
ec_map = pd.read_csv('resources/eurocrops_map_wcr_edition.csv', index_col='ec_code')

tmap = ec_map[['{}_label'.format(label_columns[-1]),'{}_name'.format(label_columns[-1])]].drop_duplicates()
tmap = dict(tmap.to_numpy())

In [ ]:
# read parquet files, assign hierarchical labels and do small clean-ups 
tpath = '/vitodata/worldcereal/features/benchmarking_features/calval/'
freq = 'monthly'

data_df = process_raw_features_input_df(
    '{}rawts-{}_calval.parquet'.format(tpath, freq),
    wc2ec_map,
    ec_map,
    label_columns,
    add_countries=True
    )

# data_df = data_df[data_df['cropland_wc']==1]
data_df = data_df[data_df['country_code'].notna()]
data_df = data_df[data_df['country_code']!='DMA']
gc.collect()

best_cropland_model_preds = pd.read_csv('/home/cbutsko/Desktop/cbutsko_experiments/country_stratified_split_finetuned_presto.csv')
best_cropland_model_preds.set_index('sample_id', inplace=True)
data_df['crop_prob'] = data_df.index.map(best_cropland_model_preds['crop_prob'])

In [ ]:
data_df.shape

In [ ]:
data_df[['country_code','year']].value_counts().sort_index().iloc[290:310]

In [ ]:
data_df[data_df['country_code']=='USA']['croptype'].map(tmap).value_counts()

In [ ]:
data_df[['ref_id','year']].value_counts().iloc[:20]

In [ ]:
data_df[data_df['country_code']=='FRA']['year'].value_counts()

In [ ]:
data_df.columns[-20:]

In [ ]:
# add pre-computed presto features 
# presto_df = pd.read_parquet('{}pretrainedprestofts-{}_calval.parquet'.format(tpath,freq))
presto_df = pd.read_parquet('{}prestofts-{}_calval.parquet'.format(tpath,freq))

presto_df.set_index('sample_id', inplace=True)
presto_emb_feats = [xx for xx in presto_df.columns if 'presto_ft' in xx]

data_df = pd.concat([data_df,presto_df[presto_emb_feats]], join='inner', axis=1)
del presto_df
gc.collect()

### SatCLIP embeddings

1. Get the embeddings themselves for train samples

In [ ]:
# again, here goes either path to a model, or git submodule can be created 🤷‍♀️
model_path = '/home/cbutsko/Desktop/cbutsko_experiments/satclip-resnet18-l10.ckpt'
model = get_satclip(model_path, device='cpu')

In [ ]:
satclip_df = prepare_satclip_embeddings(model, data_df)
satclip_emb_feats = [xx for xx in satclip_df.columns if 'satclip_ft' in xx]
data_df = pd.concat([data_df,satclip_df[satclip_emb_feats]], join='inner', axis=1)

del satclip_df
gc.collect()

2. Fit PCA on (part of) data, transform SatCLIP embeddings into PCs and append them to main dataframes

In [ ]:
n_rows = data_df.shape[0]
n_pcs = 32
satclip_pca_feats = ['satclip_PC{}'.format(jj) for jj in range(n_pcs)]

satclip_pca_model = PCA(n_components=n_pcs)
scaler = StandardScaler()
scaler.fit(data_df[satclip_emb_feats].sample(n=int(0.7*n_rows)).values)
satclip_emb_norm = scaler.transform(data_df[satclip_emb_feats].values)
satclip_pca_model.fit(satclip_emb_norm[np.random.randint(n_rows, size=int(0.7*n_rows)),:])
gc.collect()
satclip_pca = satclip_pca_model.transform(satclip_emb_norm)
satclip_pca_df = pd.DataFrame(
    satclip_pca, 
    columns=satclip_pca_feats, 
    index=data_df.index)

data_df = pd.concat([data_df,satclip_pca_df], join='inner', axis=1)

del satclip_pca_df
gc.collect()

### Classification

In [ ]:
# group feature names for easier use
n_ts = 12

optical12_feats = [xx for xx in data_df.columns if re.search(r'OPTICAL.*ts({})-'.format('|'.join(map(str, list(range(n_ts))))), xx)]
sar12_feats = [xx for xx in data_df.columns if re.search(r'SAR.*ts({})-'.format('|'.join(map(str, list(range(n_ts))))), xx)]
temp12_feats  = [xx for xx in data_df.columns if re.search(r'METEO-temp.*ts({})-'.format('|'.join(map(str, list(range(n_ts))))), xx)]
prcp12_feats  = [xx for xx in data_df.columns if re.search(r'METEO-precip.*ts({})-'.format('|'.join(map(str, list(range(n_ts))))), xx)]
dem_feats = ['DEM-alt-20m', 'DEM-slo-20m']
latlon_feats = ['lat','lon']
biome_feats = [xx for xx in data_df.columns if 'biome' in xx]

satclip_emb_feats = [xx for xx in data_df.columns if 'satclip_ft' in xx]
presto_emb_feats = [xx for xx in data_df.columns if 'presto_ft' in xx]

satclip_pca_feats = [xx for xx in data_df.columns if 'satclip_PC' in xx]

In [ ]:
for ts in range(n_ts):
    B04 = data_df['OPTICAL-B04-ts{}-10m'.format(ts)]
    B08 = data_df['OPTICAL-B08-ts{}-10m'.format(ts)]
    _ndvi = (B08 - B04) / (B08 + B04)
    _ndvi[_ndvi.abs()>1] = np.nan
    _ndvi[_ndvi.abs()==0] = np.nan
    data_df['NDVI-ts{}-10m'.format(ts)] = _ndvi
    del B04,B08,_ndvi
    gc.collect()

ndvi_feats = [xx for xx in data_df.columns if re.search(r'NDVI.*ts({})-'.format('|'.join(map(str, list(range(n_ts))))), xx)]

In [ ]:
data_df['n_peaks'] = data_df[ndvi_feats].apply(lambda xx: find_peaks(xx, prominence=0.25, distance=3)[0].shape[0], axis=1)
data_df['valid_date_ind'] = ((data_df['valid_date'] - data_df['start_date']).dt.days/30).astype(int)
data_df['valid_date_doy'] = data_df['valid_date'].dt.dayofyear

In [ ]:
data_df['valid_peak_ind'] = np.nan
seasons_mask = pd.DataFrame(False, index=data_df.index, columns=data_df.columns)
pbar = tqdm(total=len(data_df[data_df['n_peaks']>0]))
for sample_id in data_df[data_df['n_peaks']>0].index:
    pbar.update(1)
    tobs = data_df.loc[sample_id]
    peaks = find_peaks(tobs[ndvi_feats], prominence=0.25, distance=3)
    valid_peak_ind = np.argmin(np.abs(peaks[0] - tobs['valid_date_ind']))
    
    data_df.at[sample_id,'valid_peak_ind'] = peaks[0][valid_peak_ind]

    season_start_ts = peaks[1]['left_bases'][valid_peak_ind]
    season_end_ts = peaks[1]['right_bases'][valid_peak_ind]
    cols_to_null_p1 = [xx for xx in data_df.columns if any(ts in xx for ts in ['-ts{}-'.format(ii) for ii in range(season_start_ts)])]
    cols_to_null_p2 = [xx for xx in data_df.columns if any(ts in xx for ts in ['-ts{}-'.format(ii) for ii in range(season_end_ts,12)])]
    cols_to_null = cols_to_null_p1 + cols_to_null_p2
    seasons_mask.loc[sample_id][cols_to_null] = True

data_df['valid_peak_ind'].fillna(data_df['valid_date_ind'], inplace=True)

In [ ]:
data_df[data_df['CROPTYPE_NAME']=='Maize'][['country_code','n_peaks']].value_counts().sort_index().reset_index()

In [ ]:
data_df['valid_peak_ind'].value_counts()

In [ ]:
crop = 'Maize'
data_df['is_crop'] = data_df['CROPTYPE_NAME']==crop

In [ ]:
data_df['CROPTYPE_NAME'].value_counts().iloc[:20]

In [ ]:
data_df[data_df['CROPTYPE_NAME']=='Maize']['country_code'].value_counts().iloc[:20]

In [ ]:
data_df[
    (data_df['CROPTYPE_NAME']=='Maize') & 
    (data_df['crop_prob']<0.5)
    ].groupby('country_code')['n_peaks'].value_counts().reset_index().to_csv('/home/cbutsko/Desktop/cbutsko_experiments/maize_predicted_not_crop_peaks.csv')

In [ ]:
data_df.groupby(['country_code','n_peaks'])['is_crop'].agg(['count','sum']).to_csv('/home/cbutsko/Desktop/cbutsko_experiments/maize_stats.csv')

In [ ]:
data_df[data_df['n_peaks']>1]['country_code'].value_counts().iloc[20:40]

In [ ]:
data_df[
    (data_df['country_code']=='IDN') &
    (data_df['n_peaks']>1)
    ]['CROPTYPE_NAME'].value_counts().iloc[:20]

In [ ]:
sample_size = 100
n_peaks = 2
country = 'ETH'
crop = 'Maize'
    
_ndvi_subset = data_df[
    (data_df['CROPTYPE_NAME']==crop) & 
    (data_df['country_code']==country) &
    (data_df['n_peaks']==n_peaks)
    # (data_df['crop_prob']>0.7)
    ]
n_points = len(_ndvi_subset)
if n_points>sample_size:
    ndvis = _ndvi_subset.sample(n=sample_size).transpose()
else:
    ndvis = _ndvi_subset.transpose()
    sample_size = n_points
    
# ndvis.loc[ndvi_feats].fillna(method='ffill', axis=0, inplace=True)

ndvis.loc[ndvi_feats].plot(figsize=(15,9), legend=None)
for ii in range(ndvis.shape[-1]): 
    plt.axvline(ndvis.loc['valid_date_ind'].iloc[ii], linewidth=0.8) 
# plt.title(f"""
#     {crop}, {country} 
#     Random {sample_size} samples
#     (vertical lines are valid_date)
# """)
# plt.savefig('/home/cbutsko/Desktop/cbutsko_experiments/crop_charts/{}_{}.png'.format(crop,country), dpi=300)
plt.title(f"""
    {crop}, {country} 
    Random {sample_size} samples with {n_peaks} detected peaks
    (vertical lines are valid_date)
""")
plt.savefig('/home/cbutsko/Desktop/cbutsko_experiments/crop_charts/{}_{}_peaks={}.png'.format(crop,country,n_peaks), dpi=300)

In [ ]:
sample_size = 100
n_peaks = 3
country = 'MOZ'
crop = 'Winter wheat'

for predicted_as in ['crop','not_crop']:
    
    if predicted_as=='crop':
        _ndvi_subset = data_df[
            (data_df['CROPTYPE_NAME']==crop) & 
            (data_df['country_code']==country) &
            # (data_df['n_peaks']==n_peaks)
            (data_df['crop_prob']>0.7)
            ]
    else:
        _ndvi_subset = data_df[
            (data_df['CROPTYPE_NAME']==crop) & 
            (data_df['country_code']==country) &
            (data_df['crop_prob']<0.3)
            ]

    n_points = len(_ndvi_subset)
    if n_points>sample_size:
        ndvis = _ndvi_subset.sample(n=sample_size).transpose()
    else:
        ndvis = _ndvi_subset.transpose()
        sample_size = n_points
        
    # ndvis.loc[ndvi_feats].fillna(method='ffill', axis=0, inplace=True)

    ndvis.loc[ndvi_feats].plot(figsize=(15,9), legend=None)
    for ii in range(ndvis.shape[-1]): 
        plt.axvline(ndvis.loc['valid_date_ind'].iloc[ii], linewidth=0.8) 
    plt.title(f"""
        {crop}, {country} 
        Random {sample_size} samples confidently predicted as {predicted_as}
        (vertical lines are valid_date)
    """)
    plt.savefig('/home/cbutsko/Desktop/cbutsko_experiments/crop_charts/{}_{}_predicted_{}.png'.format(crop,country,predicted_as), dpi=300)
    # plt.savefig('/home/cbutsko/Desktop/cbutsko_experiments/crop_charts/{}_{}_predicted_crop.png'.format(crop,country), dpi=300)

    # plt.title(f"""
    #     {crop}, {country} 
    #     Random {sample_size} samples with {n_peaks} detected peaks
    #     (vertical lines are valid_date)
    # """)
    # plt.savefig('/home/cbutsko/Desktop/cbutsko_experiments/crop_charts/{}_{}_peaks={}.png'.format(crop,country,n_peaks), dpi=300)

In [ ]:
attr_lst = ['ref_id','lat','lon','CROPTYPE_NAME','country_code','n_peaks','valid_date','start_date','end_date','crop_prob']
data_df[data_df['crop_prob']<0.3][attr_lst].to_csv('/home/cbutsko/Desktop/cbutsko_experiments/suspicious_cropland_predicted_as_not_crop.csv')

In [ ]:
data_df[data_df['ref_id']=='2018_EU_LUCAS']['CROPTYPE_NAME'].value_counts()

In [ ]:
data_df[data_df['crop_prob']<0.3][attr_lst].shape

In [ ]:
data_df.shape

In [ ]:
25196/241961

In [ ]:
from matplotlib import colors as mcolors
import random
colors = list(dict(mcolors.BASE_COLORS, **mcolors.CSS4_COLORS).keys())
random.shuffle(colors)

In [ ]:
country = 'ESP'
n_peaks = 2

_ndvi_subset = data_df[
    (data_df['country_code']==country) &
    (data_df['CROPTYPE_NAME']==crop) &
    (data_df['n_peaks']==n_peaks)
    ]
    
n_points = len(_ndvi_subset)
if n_points>sample_size:
    ndvis = _ndvi_subset.sample(n=sample_size)[ndvi_feats].transpose()
else:
    ndvis = _ndvi_subset[ndvi_feats].transpose()

In [ ]:
tind = np.random.choice(range(sample_size))

tndvi = ndvis.iloc[:,tind]
peaks = find_peaks(tndvi, prominence=0.25, distance=3)
tndvi.plot(figsize=(12,5))
if peaks[0].shape[0]>0:
    for ii in range(peaks[0].shape[0]):
        plt.axvline(peaks[0][ii], c=colors[ii], linewidth=1.2)
        plt.axvline(peaks[1]['left_bases'][ii], c=colors[ii], linewidth=0.8, linestyle='dashed')
        plt.axvline(peaks[1]['right_bases'][ii], c=colors[ii], linewidth=0.8, linestyle='dashed')

In [ ]:
# Define features to use, label, year and/or AEZ to leave out
label_col = 'is_crop'
features_list = optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats

In [ ]:
data_df_bckp = data_df.copy()

In [ ]:
label_col = label_columns[-1]

In [ ]:
tmap = ec_map[['{}_label'.format(label_columns[-1]),'{}_name'.format(label_columns[-1])]].drop_duplicates()
tmap = dict(tmap.to_numpy())
tmap[120000000] = 'permanent_crops'
tmap[110601000] = 'oilseeds_mixture'

In [ ]:
data_df = data_df.sample(frac=1)
n_splits = 3
split_inds = np.array_split(range(len(data_df)), n_splits)
features_list = optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats
# optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats
#  + ['valid_date_doy','valid_peak_ind','valid_date_ind','n_peaks']
# + ['valid_date_doy']
# + ['valid_date_ind']
# + ['valid_peak_ind']
# + ['n_peaks']
# presto_emb_feats

In [ ]:
preds_df = pd.DataFrame()
for tsplit in range(n_splits):
    trn_inds = [x for xs in [xx for ii,xx in enumerate(split_inds) if ii!=tsplit]  for x in xs]
    val_inds = split_inds[tsplit]

    trn_df = data_df.iloc[trn_inds]
    val_df = data_df.iloc[val_inds]
    
    trn_df = trn_df[trn_df['crop_prob']>0.7]

    X_trnval_df = trn_df[features_list]
    y_trnval_df = trn_df[label_col]
    X_tst_df = val_df[features_list]
    y_tst_df = val_df[label_col]

    # initialize and train the model
    model = CatBoostClassifier(
        iterations=500, 
        depth=8,
        eval_metric='TotalF1',
        learning_rate=0.1,
        # l2_leaf_reg=150,
        verbose=50,
        random_seed=42,
        )

    model.fit(X_trnval_df, y_trnval_df)
    # probs = model.predict_proba(X_tst_df)

    # true = y_tst_df.reset_index(drop=True)[label_col]
    pred = pd.Series(model.predict(X_tst_df).flatten())
    # pred = model.predict(X_tst_df).flatten()
    # if y_tst_df.nunique()==2:
    #     pred = np.array([xx=='True' for xx in pred])

    # create dataframe with predictions and useful attributes
    _preds_df = pd.DataFrame([y_tst_df.values, pred]).transpose()
    _preds_df.columns = ['true','pred']
    # _preds_df['crop_prob'] = probs[:,1]
    _preds_df.set_index(data_df.iloc[val_inds].index, inplace=True)
    attr_lst = ['ref_id','lat','lon','CROPTYPE_NAME','country_code','n_peaks']
    _preds_df[attr_lst] = data_df.iloc[val_inds][attr_lst]
    preds_df = pd.concat((preds_df,_preds_df), axis=0)

    gdata_dfc.collect()

# print(f1_score(preds_df['true'], preds_df['pred'], average='macro'))

In [ ]:
y_trnval_df.map(tmap).value_counts()

In [ ]:
features_list = optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats + ['valid_date_doy','valid_peak_ind','valid_date_ind','n_peaks']
# optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats
#  + ['valid_date_doy','valid_peak_ind','valid_date_ind','n_peaks']
# + ['valid_date_doy']
# + ['valid_date_ind']
# + ['valid_peak_ind']
# + ['n_peaks']
# presto_emb_feats

In [ ]:
trn_df, val_df = train_test_split(
    data_df,
    stratify=data_df['croptype'],
    test_size=0.3,
    random_state=42)

In [ ]:
test_country = 'EGY'
injection_size = 0.1

trn_df = data_df[data_df['country_code']!=test_country]
val_df = data_df[data_df['country_code']==test_country]

single_sample_classes = val_df['croptype'].value_counts()[val_df['croptype'].value_counts()==1].index
if len(single_sample_classes)>0:
    val_df = val_df[~val_df[label_col].isin(single_sample_classes)]

_, injection_df = train_test_split(
    val_df,
    stratify=val_df['croptype'],
    test_size=injection_size,
    random_state=42)
trn_df = pd.concat((trn_df,injection_df), axis=0)

In [ ]:
injection_df['croptype'].map(tmap).value_counts()

In [ ]:
# trn_df = trn_df[trn_df['crop_prob']>0.7]

X_trnval_df = trn_df[features_list]
y_trnval_df = trn_df[label_col].map(tmap)
X_tst_df = val_df[features_list]
y_tst_df = val_df[label_col].map(tmap)

# initialize and train the model
model = CatBoostClassifier(
    iterations=500, 
    depth=8,
    eval_metric='TotalF1',
    learning_rate=0.1,
    # l2_leaf_reg=150,
    verbose=50,
    random_seed=42,
    )

model.fit(X_trnval_df, y_trnval_df)
pred = pd.Series(model.predict(X_tst_df).flatten())

# create dataframe with predictions and useful attributes
preds_df = pd.DataFrame([y_tst_df.values, pred]).transpose()
preds_df.columns = ['true','pred']
preds_df.set_index(val_df.index, inplace=True)
attr_lst = ['ref_id','lat','lon','CROPTYPE_NAME','country_code','n_peaks']
preds_df[attr_lst] = val_df[attr_lst]

In [ ]:
print(classification_report(
    preds_df[preds_df['n_peaks']==0]['true'], 
    preds_df[preds_df['n_peaks']==0]['pred']
    )) 

In [ ]:
print(classification_report(
    preds_df[preds_df['n_peaks']==1]['true'], 
    preds_df[preds_df['n_peaks']==1]['pred']
    )) 

In [ ]:
print(classification_report(
    preds_df[preds_df['n_peaks']>1]['true'], 
    preds_df[preds_df['n_peaks']>1]['pred']
    )) 

In [ ]:
cm = confusion_matrix(preds_df['true'], preds_df['pred'])
plot_confusion_matrix(cm, np.unique(list(np.unique(preds_df['true'].unique())) + list(np.unique(preds_df['pred'].unique()))))

In [ ]:
preds_df.groupby(['n_peaks']).apply(lambda xx: pd.Series({
    'n_pixels': xx['ref_id'].count(),
    'n_maize_pixels': xx[xx['CROPTYPE_NAME']=='Maize']['ref_id'].count(),
    'f1': f1_score(xx['true'], xx['pred']),
    'precision': precision_score(xx['true'], xx['pred']),
    'recall': recall_score(xx['true'], xx['pred']),
    })).reset_index()

In [ ]:
preds_df[preds_df['n_peaks']>1].groupby(['country_code']).apply(lambda xx: pd.Series({
    'n_pixels': xx['ref_id'].count(),
    'n_maize_pixels': xx[xx['CROPTYPE_NAME']=='Maize']['ref_id'].count(),
    'f1': f1_score(xx['true'], xx['pred']),
    'precision': precision_score(xx['true'], xx['pred']),
    'recall': recall_score(xx['true'], xx['pred']),
    })).reset_index().sort_values(by='n_maize_pixels', ascending=False).iloc[:20]

In [ ]:
tres = preds_df.groupby(['country_code','n_peaks']).apply(lambda xx: pd.Series({
    'n_pixels': xx['ref_id'].count(),
    'n_maize_pixels': xx[xx['CROPTYPE_NAME']=='Maize']['ref_id'].count(),
    'f1': f1_score(xx['true'], xx['pred']),
    'precision': precision_score(xx['true'], xx['pred']),
    'recall': recall_score(xx['true'], xx['pred']),
    })).reset_index()
tres.to_csv('/home/cbutsko/Desktop/cbutsko_experiments/maize_4fold_cv_raw_ts.csv')

In [ ]:
data_df = data_df_bckp.copy()

data_df[seasons_mask] = np.nan
data_df.fillna(65535, inplace=True)

In [ ]:
features_for_aggregation = [
    'OPTICAL-B02','OPTICAL-B03','OPTICAL-B04','OPTICAL-B05',
    'OPTICAL-B06','OPTICAL-B07','OPTICAL-B08','OPTICAL-B8A',
    'OPTICAL-B11','OPTICAL-B12',
    'NDVI',
    'SAR-VV','SAR-VH',
    'METEO-temperature_mean','METEO-precipitation_flux'
    ]
tcols = [xx for xx in data_df.columns if any([kk in xx for kk in features_for_aggregation])]
data_df[data_df[tcols]>5000][tcols] = np.nan
data_df[data_df[tcols]==0][tcols] = np.nan

pbar = tqdm(total=len(features_for_aggregation)*1)
for tfeature in features_for_aggregation:
    tcols = [xx for xx in data_df.columns if tfeature in xx]
    new_cols = [
        '{}_agg_min'.format(tfeature),
        '{}_agg_mean'.format(tfeature),
        '{}_agg_max'.format(tfeature),
        '{}_agg_std'.format(tfeature),
        ]
    data_df[new_cols] = data_df[tcols].agg(['min','mean','max','std'], axis=1)
    pbar.update(1)

aggregated_features = [xx for xx in data_df.columns if '_agg_' in xx]

In [ ]:
# data_df_no_seasons_agg = data_df.copy()
data_df_seasons_agg = data_df.copy()

In [ ]:
# data_df = data_df_no_seasons_agg.copy()
# data_df = data_df_seasons_agg.copy()

# data_df = data_df_bckp.copy()
# data_df[seasons_mask] = np.nan
# data_df.fillna(65535, inplace=True)

In [ ]:
min_points_per_country = 50
# features_list = aggregated_features + ['valid_date_ind']
# ['valid_date_doy','n_peaks']
features_list = optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats + ndvi_feats + ['valid_date_ind']
# features_list = presto_emb_feats

countries_lst = data_df[data_df['CROPTYPE_NAME']==crop]['country_code'].value_counts()
countries_lst = countries_lst[countries_lst>min_points_per_country].index

preds_df = pd.DataFrame()
pbar = tqdm(total=len(countries_lst))
for test_country in countries_lst:
    pbar.update(1)

    trn_df = data_df[data_df['country_code']!=test_country]
    val_df = data_df[data_df['country_code']==test_country]

    # trn_df = trn_df[trn_df['crop_prob']>0.7]

    X_trnval_df = trn_df[features_list]
    y_trnval_df = trn_df[label_col]
    X_tst_df = val_df[features_list]
    y_tst_df = val_df[label_col]

    class_weights = {'False':1, 'True':1}

    model = CatBoostClassifier(
        iterations=500, 
        depth=8,
        eval_metric='F1',
        learning_rate=0.05,
        l2_leaf_reg=130,
        verbose=0,
        random_seed=42,
        class_weights=class_weights
        )

    model.fit(X_trnval_df, y_trnval_df)
    probs = model.predict_proba(X_tst_df)
    pred = model.predict(X_tst_df).flatten()
    if y_tst_df.nunique()==2:
        pred = np.array([xx=='True' for xx in pred])

    # create dataframe with predictions and useful attributes
    _preds_df = pd.DataFrame([y_tst_df.values, pred]).transpose()
    _preds_df.columns = ['true','pred']
    _preds_df['crop_prob'] = probs[:,1]
    _preds_df.set_index(val_df.index, inplace=True)
    attr_lst = ['ref_id','lat','lon','CROPTYPE_NAME','country_code','n_peaks']
    _preds_df[attr_lst] = val_df[attr_lst]
    preds_df = pd.concat((preds_df,_preds_df), axis=0)

    gc.collect()

tres = preds_df.groupby(['country_code','n_peaks']).apply(lambda xx: pd.Series({
    'n_pixels': xx['ref_id'].count(),
    'n_maize_pixels': xx[xx['CROPTYPE_NAME']=='Maize']['ref_id'].count(),
    'f1': f1_score(xx['true'], xx['pred']),
    'precision': precision_score(xx['true'], xx['pred']),
    'recall': recall_score(xx['true'], xx['pred']),
    })).reset_index()
tres.to_csv('/home/cbutsko/Desktop/cbutsko_experiments/maize_country_loo_raw_ts_valid_date_loc.csv')

In [ ]:
tres

In [ ]:
data_df.groupby(['country_code','n_peaks']).apply(lambda xx: pd.Series({
    'n_cropland_pixels': xx['ref_id'].count(),
    'n_confident_cropland_pixels': xx[xx['crop_prob']>0.7]['ref_id'].count(),
    })).reset_index().to_csv('/home/cbutsko/Desktop/cbutsko_experiments/maize_stats_confident_samples.csv')

In [ ]:
tlist = [1,10,20,30,40,50,100,200,250]
countries_lst = ['BEL', 'USA', 'AUT', 'FRA', 'ARG', 'ESP', 'UKR', 'CAN', 'NLD', 'DEU', 'NGA', 'RWA', 'LVA', 'POL', 'MOZ', 'UGA', 'BRA', 'ROU', 'ITA']

res_fpath = '/home/cbutsko/Desktop/cbutsko_experiments/maize_presto_emb_test_country_injection.csv'
# res_fpath = '/home/cbutsko/Desktop/cbutsko_experiments/maize_raw_ts_test_country_injection.csv'
if os.path.isfile(res_fpath):
    maize_injection_res = pd.read_csv(res_fpath)
    if 'Unnamed: 0' in maize_injection_res.columns:
        maize_injection_res.drop('Unnamed: 0', axis=1, inplace=True)
    computed_countries = maize_injection_res['country'].unique()
    countries_lst = [xx for xx in countries_lst if xx not in computed_countries]
else: 
    maize_injection_res = pd.DataFrame()

pbar = tqdm(total=len(tlist)*len(countries_lst))
for test_country in countries_lst:
    for n_test_samples in tlist:
        pbar.update(1)
        trn_df = data_df[data_df['country_code']!=test_country]
        val_df = data_df[data_df['country_code']==test_country]

        if (val_df[label_col]).sum()<n_test_samples: continue

        test_injection_df = val_df[val_df[label_col]].sample(n=n_test_samples, random_state=42)
        trn_df = pd.concat((trn_df,test_injection_df), axis=0)

        trn_df = trn_df[trn_df['crop_prob']>0.7]

        # features_list = aggregated_features + ['valid_date_ind','n_peaks','valid_date_doy']
        features_list = optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats + ndvi_feats
        # features_list = presto_emb_feats
        # features_list = ['valid_date_ind','n_peaks'] + optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats + ndvi_feats

        X_trnval_df = trn_df[features_list]
        y_trnval_df = trn_df[label_col]
        X_tst_df = val_df[features_list]
        y_tst_df = val_df[label_col]

        class_weights = {'False':1, 'True':1}

        model = CatBoostClassifier(
            iterations=500, 
            depth=8,
            eval_metric='F1',
            learning_rate=0.05,
            l2_leaf_reg=130,
            verbose=0,
            random_seed=42,
            class_weights=class_weights
            )

        model.fit(X_trnval_df, y_trnval_df)
        probs = model.predict_proba(X_tst_df)
        pred = model.predict(X_tst_df).flatten()
        if y_tst_df.nunique()==2:
            pred = np.array([xx=='True' for xx in pred])

        # create dataframe with predictions and useful attributes
        preds_df = pd.DataFrame([y_tst_df.values, pred]).transpose()
        preds_df.columns = ['true','pred']
        # preds_df['crop_prob'] = probs[:,1]
        preds_df.set_index(val_df.index, inplace=True)
        # attr_lst = ['ref_id','lat','lon','CROPTYPE_NAME','country_code','n_peaks','crop_prob']
        # preds_df[attr_lst] = val_df[attr_lst]

        gc.collect()

        # tres = preds_df.groupby(['country_code','n_peaks']).apply(lambda xx: pd.Series({
        #     'n_pixels': xx['ref_id'].count(),
        #     'n_maize_pixels': xx[xx['CROPTYPE_NAME']=='Maize']['ref_id'].count(),
        #     'f1': f1_score(xx['true'], xx['pred']),
        #     'precision': precision_score(xx['true'], xx['pred']),
        #     'recall': recall_score(xx['true'], xx['pred']),
        #     })).reset_index()

        _f1 = f1_score(preds_df['true'], preds_df['pred'])
        tres = pd.DataFrame([test_country,n_test_samples,_f1]).transpose()
        tres.columns = ['country','n_test_samples','f1']
        maize_injection_res = pd.concat((maize_injection_res,tres), axis=0)
        maize_injection_res.to_csv(res_fpath)

In [ ]:
res_fpath = '/home/cbutsko/Desktop/cbutsko_experiments/maize_presto_emb_test_country_injection.csv'
# res_fpath = '/home/cbutsko/Desktop/cbutsko_experiments/maize_raw_ts_test_country_injection.csv'
if os.path.isfile(res_fpath):
    maize_injection_res = pd.read_csv(res_fpath)
    if 'Unnamed: 0' in maize_injection_res.columns:
        maize_injection_res.drop('Unnamed: 0', axis=1, inplace=True)

In [ ]:
maize_injection_res['n_test_samples'] = maize_injection_res['n_test_samples'].astype(int)
maize_injection_res['f1'] = maize_injection_res['f1'].astype(float)

In [ ]:
missing_piece = pd.DataFrame([
['POL',200,np.nan],
['MOZ',200,np.nan],
['UGA',200,np.nan],
['BRA',200,np.nan],
['ROU',200,np.nan],
['ITA',200,np.nan],
['POL',250,np.nan],
['MOZ',250,np.nan],
['UGA',250,np.nan],
['BRA',250,np.nan],
['ROU',250,np.nan],
['ITA',250,np.nan]
], columns=maize_injection_res.columns)
maize_injection_res = pd.concat((maize_injection_res,missing_piece), axis=0)
maize_injection_res.reset_index(drop=True, inplace=True)

In [ ]:
maize_injection_res['n_test_samples'] = maize_injection_res['n_test_samples'].astype(int)

In [ ]:
maize_injection_res = maize_injection_res.pivot(index='n_test_samples', columns='country', values='f1')
maize_injection_res[maize_injection_res==0] = np.nan

In [ ]:
maize_injection_res

In [ ]:
maize_injection_res.plot(figsize=(12,5))
plt.legend(loc='right')
plt.xlabel('n injected samples')
plt.title("""
Binary maize F1 score change depending on "injection" size from unseen country
Finetuned Presto embeddings
""")

In [ ]:
test_country = 'ARG'

trn_df = data_df.sample(frac=0.7)
val_df = data_df[data_df['country_code']==test_country]

trn_df = trn_df[trn_df['crop_prob']>0.7]

features_list = optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats + ndvi_feats + ['valid_date_ind']

X_trnval_df = trn_df[features_list]
y_trnval_df = trn_df[label_col]
X_tst_df = val_df[features_list]
y_tst_df = val_df[label_col]

class_weights = {'False':1, 'True':1}

model = CatBoostClassifier(
    iterations=500, 
    depth=8,
    eval_metric='F1',
    learning_rate=0.05,
    l2_leaf_reg=130,
    verbose=250,
    random_seed=42,
    class_weights=class_weights
    )

model.fit(X_trnval_df, y_trnval_df)
probs = model.predict_proba(X_tst_df)
pred = model.predict(X_tst_df).flatten()
if y_tst_df.nunique()==2:
    pred = np.array([xx=='True' for xx in pred])

# create dataframe with predictions and useful attributes
preds_df = pd.DataFrame([y_tst_df.values, pred]).transpose()
preds_df.columns = ['true','pred']
preds_df.set_index(val_df.index, inplace=True)
gc.collect()

print(f1_score(preds_df['true'], preds_df['pred']))

In [ ]:
trn_df = data_df[data_df['country_code']!=test_country]
val_df = data_df[data_df['country_code']==test_country]
test_injection_df = val_df[val_df[label_col]].sample(n=n_test_samples)

In [ ]:
test_country = 'UKR'
tlist = [1,10,20,30,40,50,100,200,250,300,350,400,450,500,550,600,650,700,750,800,900,1000,1100,1200]

tres = []
pbar = tqdm(total=len(tlist))
for n_test_samples in tlist:
    pbar.update(1)
    trn_df = data_df[data_df['country_code']!=test_country]
    val_df = data_df[data_df['country_code']==test_country]

    if ((val_df[label_col]).sum()<n_test_samples) | ((~val_df[label_col]).sum()<n_test_samples): continue

    test_injection_df = val_df[val_df[label_col]].sample(n=n_test_samples)
    trn_df = pd.concat((trn_df,test_injection_df), axis=0)
    trn_df = pd.concat((trn_df,val_df[~val_df[label_col]].sample(n=n_test_samples)), axis=0)

    trn_df = trn_df[trn_df['crop_prob']>0.7]

    features_list = optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats + ndvi_feats + ['valid_date_ind']
    X_trnval_df = trn_df[features_list]
    y_trnval_df = trn_df[label_col]
    X_tst_df = val_df[features_list]
    y_tst_df = val_df[label_col]

    model = CatBoostClassifier(
        iterations=500, 
        depth=8,
        eval_metric='F1',
        learning_rate=0.05,
        l2_leaf_reg=130,
        verbose=0,
        random_seed=42
        )

    model.fit(X_trnval_df, y_trnval_df)
    pred = model.predict(X_tst_df).flatten()
    if y_tst_df.nunique()==2:
        pred = np.array([xx=='True' for xx in pred])

    # create dataframe with predictions and useful attributes
    preds_df = pd.DataFrame([y_tst_df.values, pred]).transpose()
    preds_df.columns = ['true','pred']
    _f1 = f1_score(preds_df['true'], preds_df['pred'])
    tres.append([n_test_samples, _f1])

In [ ]:
pd.DataFrame(tres)

In [ ]:
print(classification_report(
    preds_df['true'], 
    preds_df['pred'], 
    target_names=['not_maize','maize'])) 

In [ ]:
print(classification_report(
    preds_df[preds_df['crop_prob']>0.7]['true'], 
    preds_df[preds_df['crop_prob']>0.7]['pred'], 
    target_names=['not_maize','maize'])) 

In [ ]:
tres[tres['n_maize_pixels']>10].sort_values(by=['country_code','n_peaks'], ascending=True).iloc[:20]

In [ ]:
tres[tres['n_maize_pixels']>10].sort_values(by=['country_code','n_peaks'], ascending=True).iloc[:20]

In [ ]:
tres[tres['n_maize_pixels']>10].sort_values(by=['n_maize_pixels'], ascending=False)

In [ ]:
tres[tres['n_maize_pixels']>10].sort_values(by=['n_maize_pixels'], ascending=False)